# JEPA News Recommendation System - Stage 1 Colab Runner

This notebook runs the JEPA-only product demo on Google Colab: FastAPI backend plus optional Vite frontend. It keeps Stage 1 artifacts on the Colab runtime machine under `/content/jepa-news-rec`.

## 0. Runtime setup

Use **Runtime -> Change runtime type -> GPU** if you want CUDA inference. CPU also works for smoke testing, but scoring all articles is slower.

In [13]:
import os

REPO_URL = "https://github.com/AayushChhabra42/jepa-news-rec.git"
REPO_DIR = "/content/jepa-news-rec"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")

%cd {REPO_DIR}
!git pull --ff-only || true

In [ ]:
# Backend dependencies. pydantic-settings/FastAPI may not be present in older repo snapshots.
%pip install -q -r requirements.txt
%pip install -q fastapi "uvicorn[standard]" pydantic-settings scikit-learn pyngrok

# Frontend dependencies. This can take a few minutes on Colab.
!cd frontend && npm install

## 1. Provide Stage 1 artifacts

The API will not start without a real JEPA checkpoint. Keep artifacts inside the Colab runtime under `/content/jepa-news-rec`. Expected files:

- `checkpoints/jepa_best.pt`
- optional `checkpoints/finetuned_model.pt`
- `data/processed/processed_data.pkl`, or split files under `data/processed/`

You can either upload a zip/tar artifact bundle into the runtime, or run the optional training cells below to create the artifacts directly on the Colab machine.

In [ ]:
from pathlib import Path

ARTIFACT_ROOT = Path('/content/jepa-news-rec')
Path('checkpoints').mkdir(exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)

print('Artifacts will live under:', ARTIFACT_ROOT)
print('Checkpoint dir:', ARTIFACT_ROOT / 'checkpoints')
print('Processed data dir:', ARTIFACT_ROOT / 'data' / 'processed')

In [ ]:
# Optional: upload an artifact bundle from your machine into this Colab runtime.
# The archive should contain checkpoints/ and data/processed/ folders.
# Leave this cell unrun if you are going to create artifacts from scratch below.

from google.colab import files
from pathlib import Path
import shutil
import tarfile
import zipfile

uploaded = files.upload()
for name in uploaded:
    archive = Path(name)
    print('Extracting', archive)
    if archive.suffix == '.zip':
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('/content/jepa-news-rec')
    elif archive.suffixes[-2:] in [['.tar', '.gz']] or archive.suffix in ['.tgz', '.tar']:
        with tarfile.open(archive) as tf:
            tf.extractall('/content/jepa-news-rec')
    else:
        raise ValueError(f'Unsupported archive type: {archive}')

!find checkpoints data/processed -maxdepth 2 -type f | sort | sed -n '1,80p'

In [8]:
from pathlib import Path

required = [
    Path('checkpoints/jepa_best.pt'),
]
processed_options = [
    Path('data/processed/processed_data.pkl'),
    Path('data/processed/news.pkl'),
]

missing = [str(path) for path in required if not path.exists()]
if not any(path.exists() for path in processed_options):
    missing.append('data/processed/processed_data.pkl or split processed files')

if missing:
    raise FileNotFoundError('Missing Stage 1 artifacts: ' + ', '.join(missing))

print('Stage 1 artifacts found.')
!find checkpoints data/processed -maxdepth 2 -type f | sort | sed -n '1,80p'

## Optional: build artifacts from scratch

Only run these cells if you do not already have artifacts. They can take a long time and require Kaggle credentials for MIND-small.

In [ ]:
# Optional: set Kaggle credentials if you need to download MIND-small from scratch.
# Prefer uploading /root/.kaggle/kaggle.json through the Colab file browser.
os.environ['KAGGLE_USERNAME'] = ''
os.environ['KAGGLE_KEY'] = ''

!python data/download.py --dataset mind-small --source kaggle
!python data/preprocess.py --dataset mind-small
!python scripts/train_simcse.py --epochs 5
!python scripts/train_jepa.py --epochs 7
!python scripts/finetune.py --epochs 3

In [ ]:
!python baselines/xgboost_ranker.py

## 2. Start the Stage 1 FastAPI backend

In [14]:
import os
import torch

os.environ['JEPA_PROCESSED_DIR'] = '/content/jepa-news-rec/data/processed'
os.environ['JEPA_CHECKPOINT_PATH'] = '/content/jepa-news-rec/checkpoints/jepa_best.pt'
os.environ['JEPA_FINETUNED_CHECKPOINT_PATH'] = '/content/jepa-news-rec/checkpoints/finetuned_model.pt'
os.environ['JEPA_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['JEPA_TOP_K_DEFAULT'] = '50'
os.environ['JEPA_CORS_ORIGINS'] = '["*"]'

print('JEPA_DEVICE =', os.environ['JEPA_DEVICE'])

In [15]:
import subprocess
import time
import requests
from pathlib import Path

api_log = open('/content/jepa_api.log', 'w')
api_proc = subprocess.Popen(
    ['uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/jepa-news-rec',
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(10):
    try:
        response = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
        if response.ok:
            print('API ready:', response.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print(Path('/content/jepa_api.log').read_text()[-4000:])
    raise RuntimeError('API did not become ready. See /content/jepa_api.log')

In [16]:
import requests

base = 'http://127.0.0.1:8000/api'
users = requests.get(f'{base}/users', params={'page': 1, 'page_size': 5}).json()
print(users)

user_id = users['users'][0]
profile = requests.get(f'{base}/users/{user_id}/profile').json()
print('profile user:', profile['user_id'], 'history items:', len(profile['history']))

hidden = requests.get(
    f'{base}/users/{user_id}/recommendations',
    params={'top_k': 10, 'stage': 'jepa', 'label_reveal': 'false'},
).json()
print('hidden labels:', [item['label'] for item in hidden['recommendations'][:3]])
print('metrics:', hidden['metrics'])

revealed = requests.get(
    f'{base}/users/{user_id}/recommendations',
    params={'top_k': 10, 'stage': 'jepa', 'label_reveal': 'true'},
).json()
print('revealed labels:', [item['label'] for item in revealed['recommendations'][:3]])

## 3. Open the app from Colab

Use pyngrok tunnels for the API and Vite frontend. If ngrok asks for an auth token, add it in the next cell.

In [19]:
# Optional but often required by ngrok:
from pyngrok import ngrok
auth_token = os.getenv('NGROK_AUTH_TOKEN', '')
ngrok.set_auth_token(auth_token)

In [20]:
from pyngrok import ngrok
import requests

old_api_tunnel = globals().get('api_tunnel')
if old_api_tunnel:
    try:
        ngrok.disconnect(old_api_tunnel.public_url)
    except Exception:
        pass

api_tunnel = ngrok.connect(8000, bind_tls=True)
BACKEND_PUBLIC_URL = api_tunnel.public_url

health_url = BACKEND_PUBLIC_URL + '/api/health'
health = requests.get(health_url, headers={'ngrok-skip-browser-warning': 'true'}, timeout=10)
print('Backend API:', BACKEND_PUBLIC_URL)
print('Health URL:', health_url)
print('Health response:', health.text[:300])
health.raise_for_status()
assert health.headers.get('content-type', '').startswith('application/json'), 'Backend tunnel is not returning JSON.'
assert health.json()['status'] == 'ok'


In [21]:
import subprocess
import time
from pathlib import Path

old_front_proc = globals().get('front_proc')
if old_front_proc and old_front_proc.poll() is None:
    old_front_proc.terminate()
    time.sleep(2)

# Colab can keep old Vite servers alive after interrupted cells. Clear them so ngrok
# does not keep forwarding to an older server with stale allowedHosts settings.
subprocess.run(['bash', '-lc', "pkill -f 'vite.*5173' || true; fuser -k 5173/tcp || true"], check=False)
time.sleep(2)

frontend_env = os.environ.copy()
frontend_env['VITE_API_BASE_URL'] = '/api'
print('Frontend will call API through Vite proxy:', frontend_env['VITE_API_BASE_URL'])

# Write this in Colab too, in case the runtime has an older repo copy.
Path('/content/jepa-news-rec/frontend/vite.config.js').write_text('''import react from "@vitejs/plugin-react";\nimport { defineConfig } from "vite";\n\nexport default defineConfig({\n  plugins: [react()],\n  server: {\n    host: "0.0.0.0",\n    allowedHosts: true,\n    hmr: false,\n    proxy: {\n      "/api": {\n        target: "http://127.0.0.1:8000",\n        changeOrigin: true\n      }\n    }\n  }\n});\n''')

front_log = open('/content/jepa_frontend.log', 'w')
front_proc = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173', '--strictPort'],
    cwd='/content/jepa-news-rec/frontend',
    env=frontend_env,
    stdout=front_log,
    stderr=subprocess.STDOUT,
)
time.sleep(8)
print(Path('/content/jepa_frontend.log').read_text()[-3000:])
if front_proc.poll() is not None:
    raise RuntimeError('Vite frontend exited. See /content/jepa_frontend.log')

In [22]:
old_frontend_tunnel = globals().get('frontend_tunnel')
if old_frontend_tunnel:
    try:
        ngrok.disconnect(old_frontend_tunnel.public_url)
    except Exception:
        pass

frontend_tunnel = ngrok.connect(5173, bind_tls=True)
FRONTEND_PUBLIC_URL = frontend_tunnel.public_url
print('Open frontend:', FRONTEND_PUBLIC_URL)
print('The frontend serves UI and proxies /api to local FastAPI on port 8000.')


## 4. Shutdown

Run this when finished to stop background processes and close tunnels.

In [23]:
for proc_name in ['front_proc', 'api_proc']:
    proc = globals().get(proc_name)
    if proc and proc.poll() is None:
        proc.terminate()
        print('terminated', proc_name)

try:
    ngrok.kill()
    print('ngrok tunnels closed')
except Exception as exc:
    print(exc)